# Differential Expression Analysis: d0 → d2 → d5
## A Heavily-Commented Guide for New Python Users

This notebook analyzes gene expression changes across three timepoints in bovine cells.

**What we're doing:**
1. Load gene expression data from CSV files
2. Find genes that change consistently across timepoints
3. Create visualizations to understand the patterns
4. Identify biological pathways affected by these changes

## Cell 0: Install & Import Libraries

**What are libraries?**
Libraries are bundles of pre-written code that solve common problems. Instead of writing everything from scratch, we import them to use their functions.

**What each library does:**
- `pandas`: Read/manipulate data (spreadsheet-like operations)
- `numpy`: Mathematical operations on arrays of numbers
- `matplotlib` & `seaborn`: Create publication-quality plots
- `gseapy`: Query biological databases for pathway information
- `matplotlib_venn`: Draw Venn diagrams

In [ ]:
# Install packages if not already installed
# The exclamation mark (!) tells Jupyter to run a shell command
# The -q flag means "quiet" (don't print lots of installation messages)
!pip install gseapy matplotlib-venn -q

In [ ]:
# Import all the libraries we'll use in this notebook
import pandas as pd  # pandas is usually imported as 'pd' for short
import numpy as np  # numpy is usually imported as 'np' for short
import matplotlib.pyplot as plt  # pyplot is the plotting part of matplotlib
import seaborn as sns  # seaborn is built on matplotlib but makes prettier plots
from matplotlib_venn import venn2  # venn2 is a specific function for 2-set Venn diagrams
import gseapy as gp  # gseapy is for gene set enrichment (searching biological databases)
import warnings  # warnings lets us suppress non-critical messages

# This line tells Python to ignore certain non-critical warning messages
# This keeps the output cleaner
warnings.filterwarnings('ignore')

# Set plotting style (this makes our plots look nicer)
sns.set_style('whitegrid')  # Use white background with grid lines
# Set default figure size (width=12 inches, height=8 inches)
plt.rcParams['figure.figsize'] = (12, 8)

# Print a message to confirm everything loaded successfully
print('✓ All libraries imported successfully')

## Section 1: Load & Preview Data

**What we're doing here:**
1. Read the CSV (comma-separated values) files into "DataFrames" (pandas' spreadsheet-like data structure)
2. Check how many genes we have in each comparison
3. Look at a few rows to understand the data format

**About the data:**
- Each row is a gene
- Each column is a measurement (p-value, fold change, etc.)
- `p_val_adj` = how confident we are that the change is real (lower = more confident)
- `avg_log2FC` = how much the gene changed (positive = more expression, negative = less expression)

In [ ]:
# Load the CSV files into pandas DataFrames
# index_col=0 tells pandas to use the first column (gene names) as the row labels
p2vp0 = pd.read_csv('/Users/yitong/Documents/GitHub/ClaudeWorkshop/sc-RNAseq_data/DE.SC1.P2vP0.csv', index_col=0)
p5vp2 = pd.read_csv('/Users/yitong/Documents/GitHub/ClaudeWorkshop/sc-RNAseq_data/DE.SC1.P5vP2.csv', index_col=0)

# Print a header to organize our output
print('='*80)  # Print 80 equal signs to make a line
print('DATASET OVERVIEW')
print('='*80)

# len() returns how many rows are in the DataFrame
# {variable:,} means "print with comma separators" (e.g., 1,234 instead of 1234)
print(f"\nd0 → d2 (P2 vs P0): {len(p2vp0):,} genes")
print(f"d2 → d5 (P5 vs P2): {len(p5vp2):,} genes")

In [ ]:
# Show the column names (what data we have)
print(f"\nColumns: {list(p2vp0.columns)}")

# Show the first 5 rows of the d0→d2 comparison
# This lets us see what the raw data looks like
print(f"\nFirst 5 rows (d0→d2):")
print(p2vp0.head())  # .head() means "show the first 5 rows"

In [ ]:
# Now let's count how many genes are significantly changed
# A p-value of 0.05 is a common significance threshold (5% chance of false positive)
sig_threshold = 0.05

# These are BOOLEAN MASKS - they create True/False for each row
# Example: if p_val_adj < 0.05 is True for gene A but False for gene B,
# then sig_d0vd2_up[A] = True, sig_d0vd2_up[B] = False

# Genes that went UP (higher expression) in d0→d2 comparison
sig_d0vd2_up = (p2vp0['p_val_adj'] < sig_threshold) & (p2vp0['avg_log2FC'] > 0)

# Genes that went DOWN (lower expression) in d0→d2 comparison
sig_d0vd2_down = (p2vp0['p_val_adj'] < sig_threshold) & (p2vp0['avg_log2FC'] < 0)

# Same for the d2→d5 comparison
sig_d2vd5_up = (p5vp2['p_val_adj'] < sig_threshold) & (p5vp2['avg_log2FC'] > 0)
sig_d2vd5_down = (p5vp2['p_val_adj'] < sig_threshold) & (p5vp2['avg_log2FC'] < 0)

# Count how many genes meet each criteria
# .sum() on a Boolean column counts the True values
print('\nSignificant genes (adjusted p-value < 0.05):')
print(f"  d0→d2: {sig_d0vd2_up.sum():,} UP | {sig_d0vd2_down.sum():,} DOWN")
print(f"  d2→d5: {sig_d2vd5_up.sum():,} UP | {sig_d2vd5_down.sum():,} DOWN")

In [ ]:
# Find genes that appear in BOTH comparisons (overlap)
# set() is a Python data structure that stores unique items and supports set operations
# & is the "intersection" operator - it finds common items between two sets

# Convert index (gene names) to sets
overlap = set(p2vp0.index) & set(p5vp2.index)

print(f"\nGenes in both comparisons: {len(overlap):,}")

## Section 2: Volcano Plots

**What is a volcano plot?**
- X-axis: How much did the gene change? (fold change)
- Y-axis: How confident are we? (-log10 p-value)
- Dots far up and far left/right = genes we're confident changed a lot
- Dots in the middle = genes that changed very little or we're not confident about

**Why -log10(p-value)?**
- Regular p-values are tiny (0.00001) which is hard to see on plots
- -log10 transforms them to readable numbers (5)
- More confident changes = higher -log10(p-value)

In [ ]:
# Define a FUNCTION - a reusable block of code that does the same thing multiple times
# Functions take INPUTS (parameters) and produce OUTPUTS (returns)
def plot_volcano(data, title, ax):
    """
    Create a volcano plot.
    
    Parameters:
    - data: DataFrame with 'p_val_adj' and 'avg_log2FC' columns
    - title: String title for the plot
    - ax: matplotlib axis to draw on
    
    This function doesn't return anything, it just modifies 'ax' in place.
    """
    
    # Create a new column with -log10(p-value)
    # np.log10() takes the base-10 logarithm
    # The minus sign flips it so bigger numbers = more significant
    data['log10_pval'] = -np.log10(data['p_val_adj'])
    
    # Create a color list: start with gray for all genes
    # We'll change the color for significant ones
    colors = ['gray'] * len(data)  # Multiply a list to repeat it
    
    # Loop through each gene and its data
    # enumerate() gives us both the index (i) and the data (row)
    for i, (idx, row) in enumerate(data.iterrows()):
        # Check if this gene is significantly changed
        if row['p_val_adj'] < 0.05:
            # If it went up, color it red
            if row['avg_log2FC'] > 0:
                colors[i] = '#d62728'  # HTML color code for red
            # If it went down, color it blue
            else:
                colors[i] = '#1f77b4'  # HTML color code for blue
    
    # Create the scatter plot (dots at x,y positions)
    # c=colors means use our color list
    # alpha=0.6 means 60% opacity (so overlapping dots show)
    # s=30 means dot size is 30
    ax.scatter(data['avg_log2FC'], data['log10_pval'], 
              c=colors, alpha=0.6, s=30, edgecolors='none')
    
    # Add a horizontal line at the significance threshold
    # This shows where p=0.05 is
    ax.axhline(-np.log10(0.05), color='black', linestyle='--', 
               linewidth=1, alpha=0.5)
    
    # Add a vertical line at x=0 (no fold change)
    ax.axvline(0, color='black', linestyle='-', linewidth=0.5, alpha=0.3)
    
    # Find the 10 most significant genes (highest -log10 p-values)
    top_genes = data.nlargest(10, 'log10_pval')
    
    # Label each top gene on the plot
    for idx, row in top_genes.iterrows():
        # annotate() adds text labels to specific points
        # xytext=(5,5) means offset label 5 pixels right and up
        ax.annotate(idx, xy=(row['avg_log2FC'], row['log10_pval']),
                   xytext=(5, 5), textcoords='offset points', 
                   fontsize=8, alpha=0.8)
    
    # Label the axes
    ax.set_xlabel('log2 Fold Change', fontsize=11)
    ax.set_ylabel('-log10(adjusted p-value)', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    # Add grid for easier reading
    ax.grid(True, alpha=0.3)


# Now use the function we defined
# Create a figure with 2 subplots side by side
# figsize=(14, 5) means 14 inches wide, 5 inches tall
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Call our function for each comparison
plot_volcano(p2vp0, 'd0 → d2 (P2 vs P0)', axes[0])
plot_volcano(p5vp2, 'd2 → d5 (P5 vs P2)', axes[1])

# Make sure subplots don't overlap
plt.tight_layout()

# Save the figure as an image file
plt.savefig('volcano_plots.png', dpi=300, bbox_inches='tight')

# Display the plot
plt.show()

print('✓ Volcano plots saved')

## Section 3: Venn Diagrams

**What is a Venn diagram?**
- Shows overlap between two groups
- Circle 1: Genes that changed in comparison 1
- Circle 2: Genes that changed in comparison 2
- Middle (overlap): Genes that changed in BOTH comparisons
- This helps us find genes with consistent patterns across time

In [ ]:
# Extract gene names for each category
# set() creates a set data type (like a list but optimized for membership testing)
# p2vp0[sig_d0vd2_up].index gets the gene names of significantly up-regulated genes

genes_d0vd2_up = set(p2vp0[sig_d0vd2_up].index)
genes_d0vd2_down = set(p2vp0[sig_d0vd2_down].index)
genes_d2vd5_up = set(p5vp2[sig_d2vd5_up].index)
genes_d2vd5_down = set(p5vp2[sig_d2vd5_down].index)

# Create a figure with 2 plots side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Draw Venn diagram for upregulated genes
# venn2() takes a LIST of two SETS
venn2([genes_d0vd2_up, genes_d2vd5_up], 
      set_labels=('d0→d2\nUP', 'd2→d5\nUP'), ax=axes[0])
axes[0].set_title('Upregulated Genes Overlap', fontsize=12, fontweight='bold')

# Draw Venn diagram for downregulated genes
venn2([genes_d0vd2_down, genes_d2vd5_down], 
      set_labels=('d0→d2\nDOWN', 'd2→d5\nDOWN'), ax=axes[1])
axes[1].set_title('Downregulated Genes Overlap', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('venn_diagrams.png', dpi=300, bbox_inches='tight')
plt.show()

print('✓ Venn diagrams saved')

In [ ]:
# Print statistics about the overlaps
print(f"\nUPREGULATED overlap:")
# The - operator on sets gives you the DIFFERENCE (items in first set but not second)
print(f"  Only in d0→d2: {len(genes_d0vd2_up - genes_d2vd5_up)}")
print(f"  Only in d2→d5: {len(genes_d2vd5_up - genes_d0vd2_up)}")
# The & operator gives the INTERSECTION (items in both sets)
print(f"  In both: {len(genes_d0vd2_up & genes_d2vd5_up)}")

print(f"\nDOWNREGULATED overlap:")
print(f"  Only in d0→d2: {len(genes_d0vd2_down - genes_d2vd5_down)}")
print(f"  Only in d2→d5: {len(genes_d2vd5_down - genes_d0vd2_down)}")
print(f"  In both: {len(genes_d0vd2_down & genes_d2vd5_down)}")

## Section 4: Find Genes with Consistent Trends

**What we're looking for:**
- **Consistently UP**: Gene increases in BOTH d0→d2 AND d2→d5
  - These likely drive the biological change
- **Consistently DOWN**: Gene decreases in BOTH d0→d2 AND d2→d5
  - These are being suppressed as the process proceeds
- **Reversals**: Gene goes up then down (or down then up)
  - These might be transiently needed or transiently suppressed

In [ ]:
# Find genes with CONSISTENT trends
# & is the intersection operator (same as we used before)

# Genes that are UP in both comparisons
consistently_up = genes_d0vd2_up & genes_d2vd5_up

# Genes that are DOWN in both comparisons
consistently_down = genes_d0vd2_down & genes_d2vd5_down

# Genes that go UP then DOWN (peak at d2, then decline)
reversal_up_to_down = genes_d0vd2_up & genes_d2vd5_down

# Genes that go DOWN then UP (suppressed at d2, then re-activated)
reversal_down_to_up = genes_d0vd2_down & genes_d2vd5_up

# Print summary statistics
print('='*80)
print('CONSISTENT TREND ANALYSIS')
print('='*80)
print(f"\nConsistently UPREGULATED: {len(consistently_up)} genes")
print(f"Consistently DOWNREGULATED: {len(consistently_down)} genes")
print(f"UP→DOWN reversals: {len(reversal_up_to_down)} genes")
print(f"DOWN→UP reversals: {len(reversal_down_to_up)} genes")

In [ ]:
# Build a detailed table of consistently upregulated genes
# This creates a LIST (ordered collection) of DICTIONARIES (key-value pairs)

print('\n' + '='*80)
print('TOP 20: CONSISTENTLY UPREGULATED (d0→d2→d5)')
print('='*80)

up_genes = []  # Start with an empty list

# sorted() arranges items in order
# For each gene in consistently_up (sorted by fold change in descending order)
for gene in sorted(consistently_up, key=lambda x: p2vp0.loc[x, 'avg_log2FC'], reverse=True):
    # Get the fold change values from each comparison
    fc_d0vd2 = p2vp0.loc[gene, 'avg_log2FC']  # loc[] means "locate by label"
    fc_d2vd5 = p5vp2.loc[gene, 'avg_log2FC']
    
    # Create a dictionary (key: value pairs) for this gene
    up_genes.append({
        'Gene': gene,
        'd0→d2': fc_d0vd2,
        'd2→d5': fc_d2vd5,
        'Total': fc_d0vd2 + fc_d2vd5  # Sum of both changes
    })

# Convert the list of dictionaries into a DataFrame (like a spreadsheet)
up_df = pd.DataFrame(up_genes)

# Print the first 20 rows
# to_string() converts to a nicely formatted table
print(up_df.head(20).to_string(index=False))

In [ ]:
# Same process for downregulated genes
print('\n' + '='*80)
print('TOP 20: CONSISTENTLY DOWNREGULATED (d0→d2→d5)')
print('='*80)

down_genes = []

# Sort by fold change in ascending order (most negative first)
for gene in sorted(consistently_down, key=lambda x: p2vp0.loc[x, 'avg_log2FC']):
    fc_d0vd2 = p2vp0.loc[gene, 'avg_log2FC']
    fc_d2vd5 = p5vp2.loc[gene, 'avg_log2FC']
    down_genes.append({
        'Gene': gene,
        'd0→d2': fc_d0vd2,
        'd2→d5': fc_d2vd5,
        'Total': fc_d0vd2 + fc_d2vd5
    })

down_df = pd.DataFrame(down_genes)
print(down_df.head(20).to_string(index=False))

In [ ]:
# Save these results to CSV files for later use
# CSV = Comma-Separated Values (a simple text format for data)
up_df.to_csv('/Users/yitong/Documents/GitHub/ClaudeWorkshop/sc-RNAseq_data/consistently_upregulated.csv', 
              index=False)  # index=False means don't save the row numbers
down_df.to_csv('/Users/yitong/Documents/GitHub/ClaudeWorkshop/sc-RNAseq_data/consistently_downregulated.csv', 
                index=False)

print('\n✓ Saved consistently_upregulated.csv and consistently_downregulated.csv')

## Section 5: Heatmap of Top Genes

**What is a heatmap?**
- Shows numbers as colors (red = high, blue = low)
- X-axis: Different comparisons (d0→d2 and d2→d5)
- Y-axis: Different genes
- Easy to visually spot patterns

In [ ]:
# Select the top genes by total fold change
# .nlargest(30, 'Total') gets the 30 rows with highest values in 'Total' column
top_up = up_df.nlargest(30, 'Total')['Gene'].tolist()  # .tolist() converts to a simple list

# .nsmallest(30, 'Total') gets the 30 rows with LOWEST values (most negative)
top_down = down_df.nsmallest(30, 'Total')['Gene'].tolist()

# Combine the lists
top_genes_list = top_up + top_down
print(f"Creating heatmap with {len(top_genes_list)} genes (30 up + 30 down)")

In [ ]:
# Build a matrix (table) with fold change values
# This is a 2-column DataFrame: one column for each timepoint comparison
heatmap_data = pd.DataFrame({
    'd0→d2': [p2vp0.loc[g, 'avg_log2FC'] if g in p2vp0.index else 0 for g in top_genes_list],
    'd2→d5': [p5vp2.loc[g, 'avg_log2FC'] if g in p5vp2.index else 0 for g in top_genes_list]
}, index=top_genes_list)  # index= sets the row labels to gene names

# Create the heatmap
plt.figure(figsize=(6, 14))

# sns.clustermap creates a heatmap with clustering (groups similar rows together)
# cmap='RdBu_r' is the color scheme (Red-Blue reversed)
# center=0 means white = 0, red = positive, blue = negative
# col_cluster=False means don't rearrange the columns (we want d0→d2 and d2→d5 in order)
# yticklabels=True means show all gene names on the y-axis
sns.clustermap(heatmap_data, cmap='RdBu_r', center=0, 
               col_cluster=False, cbar_kws={'label': 'log2FC'},
               yticklabels=True, figsize=(8, 14))

# Add a title
plt.suptitle('Top 30 Upregulated + Top 30 Downregulated Genes', y=0.995)
plt.tight_layout()
plt.savefig('heatmap_topgenes.png', dpi=300, bbox_inches='tight')
plt.show()

print('✓ Heatmap saved')

## Section 6: Pathway Enrichment - Important Context

**What is pathway enrichment?**
We have a list of genes that changed. But what do they DO? Enrichment analysis:
1. Takes our gene list
2. Searches biological databases (GO, KEGG)
3. Finds pathways/functions that are overrepresented
4. Tells us: "Your genes are enriched in X biological process"

**About the databases:**
- GO = Gene Ontology (vocabulary of biological concepts)
- KEGG = Kyoto Encyclopedia of Genes and Genomes (biological pathways)

**Important caveat:**
This data is from bovine (Bos taurus) cells, but the annotation databases are human-centric.
This is OK because:
- ~97% of genes are conserved human-style symbols (e.g., COL1A1, MT1A)
- The biological functions are mostly conserved across mammals
- This is standard practice in genomics research

What we EXCLUDE:
- ENSBTAG genes (bovine-specific IDs with no human annotation) ~3% of data
- These can't be mapped to human gene sets so we skip them

In [ ]:
# Define a helper function to remove unmappable genes
def filter_ensbtag(gene_list):
    """
    Remove ENSBTAG genes (bovine-specific IDs) that can't map to human databases.
    
    ENSBTAG genes look like: ENSBTAG00000000123
    These have no human equivalent, so enrichment analysis won't work on them.
    """
    # Keep only genes that DON'T start with 'ENSBTAG'
    # This is a LIST COMPREHENSION - a compact way to filter a list
    filtered = [g for g in gene_list if not g.startswith('ENSBTAG')]
    
    # Print how many we removed
    print(f"Genes before filtering: {len(gene_list)}, after: {len(filtered)} (removed {len(gene_list) - len(filtered)} ENSBTAG IDs)")
    
    return filtered

# Prepare gene lists for enrichment
up_genes_list = filter_ensbtag(list(consistently_up))
down_genes_list = filter_ensbtag(list(consistently_down))

print(f"\nReady for enrichment:")
print(f"  Consistently UP: {len(up_genes_list)} genes")
print(f"  Consistently DOWN: {len(down_genes_list)} genes")

In [ ]:
# Run enrichment analysis using gseapy
# This connects to online databases and searches for pathways
# NOTE: This requires internet connection and may take 30-60 seconds

print('Running pathway enrichment... (requires internet connection)')
print('This may take 30-60 seconds...\n')

try:
    # Run enrichr for upregulated genes
    # gene_sets= specifies which databases to search
    # organism='human' tells it to use human annotations
    # outdir=None means don't save temporary files
    enr_up = gp.enrichr(
        gene_list=up_genes_list,
        gene_sets=['GO_Biological_Process_2023', 'GO_Molecular_Function_2023', 'KEGG_2021_Human'],
        organism='human',
        outdir=None
    )
    
    # Run enrichr for downregulated genes
    enr_down = gp.enrichr(
        gene_list=down_genes_list,
        gene_sets=['GO_Biological_Process_2023', 'GO_Molecular_Function_2023', 'KEGG_2021_Human'],
        organism='human',
        outdir=None
    )
    
    print('✓ Enrichment complete!')
    
# If something goes wrong, catch the error and print a message
except Exception as e:
    print(f'⚠ Error during enrichment: {e}')
    print('This often happens if internet is unavailable or Enrichr API is rate-limited.')
    enr_up = None
    enr_down = None

## Section 7: Show Enrichment Results - Upregulated Genes

In [ ]:
# Check if enrichment succeeded before trying to display results
if enr_up is not None:
    print('\n' + '='*80)
    print('TOP PATHWAYS: CONSISTENTLY UPREGULATED GENES')
    print('='*80)
    
    # Extract the results table
    # .results is an attribute of the enrichr object that contains the DataFrame
    results_up = enr_up.results
    
    # Filter to only significant results (p < 0.05)
    # This removes noise and shows only confident findings
    results_up = results_up[results_up['Adjusted P-value'] < 0.05]
    
    # Loop through each gene set database
    for gene_set in ['GO_Biological_Process_2023', 'GO_Molecular_Function_2023', 'KEGG_2021_Human']:
        print(f"\n{gene_set}:")
        
        # Filter results to just this database
        # .head(10) shows first 10 rows (most significant)
        subset = results_up[results_up['Gene_set'] == gene_set].head(10)
        
        # Check if there are any results
        if len(subset) > 0:
            # Loop through each pathway
            for idx, row in subset.iterrows():
                # Print the pathway name, p-value, and how many genes overlap
                print(f"  {row['Term']:60s} | p={row['Adjusted P-value']:.2e} | genes={row['Overlap']}")
        else:
            print(f"  No significant terms (p < 0.05)")
else:
    print('Enrichment failed — skipping results visualization')

## Section 8: Show Enrichment Results - Downregulated Genes

In [ ]:
# Same as above but for downregulated genes
if enr_down is not None:
    print('\n' + '='*80)
    print('TOP PATHWAYS: CONSISTENTLY DOWNREGULATED GENES')
    print('='*80)
    
    results_down = enr_down.results
    results_down = results_down[results_down['Adjusted P-value'] < 0.05]
    
    for gene_set in ['GO_Biological_Process_2023', 'GO_Molecular_Function_2023', 'KEGG_2021_Human']:
        print(f"\n{gene_set}:")
        subset = results_down[results_down['Gene_set'] == gene_set].head(10)
        
        if len(subset) > 0:
            for idx, row in subset.iterrows():
                print(f"  {row['Term']:60s} | p={row['Adjusted P-value']:.2e} | genes={row['Overlap']}")
        else:
            print(f"  No significant terms (p < 0.05)")
else:
    print('Enrichment failed — skipping results visualization')

## Section 9: Enrichment Dot Plots

**What is a dot plot for enrichment?**
- X-axis: Gene ratio (how many of our genes are in this pathway)
- Y-axis: Pathway names
- Dot size: How many genes overlap
- Dot color: How significant (darker = more significant)

This gives a quick visual summary of which pathways are most important.

In [ ]:
# Only make plots if enrichment succeeded
if enr_up is not None and enr_down is not None:
    
    # Define a helper function to prepare data for plotting
    def prep_dotplot_data(results_df, gene_set, top_n=15):
        """
        Filter and prepare enrichment results for dot plot.
        
        Takes results, filters to one database, and extracts the data we need for plotting.
        """
        # Filter to the specific gene set (database)
        subset = results_df[results_df['Gene_set'] == gene_set].copy()
        
        if len(subset) > 0:
            # Calculate -log10(p-value) for color intensity
            subset['log_pval'] = -np.log10(subset['Adjusted P-value'])
            
            # Extract the first number from the 'Overlap' column
            # Overlap looks like "10/200" meaning 10 genes overlap out of 200 in pathway
            # .str.split('/') splits by '/', .str[0] gets first part
            subset['gene_count'] = subset['Overlap'].str.split('/').str[0].astype(int)
            
            # Calculate gene ratio (how many of the pathway genes are in our list)
            subset['gene_ratio'] = subset['gene_count'] / subset['Overlap'].str.split('/').str[1].astype(int)
            
            # Return top 15 most significant
            return subset.nlargest(top_n, 'log_pval')
        
        return pd.DataFrame()  # Return empty DataFrame if no results
    
    # Create a 2x3 grid of subplots (2 rows, 3 columns)
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    fig.suptitle('Pathway Enrichment: Dot Plots', fontsize=14, fontweight='bold')
    
    # The three databases we searched
    gene_sets = ['GO_Biological_Process_2023', 'GO_Molecular_Function_2023', 'KEGG_2021_Human']
    
    # UPREGULATED GENES (top row)
    for i, gene_set in enumerate(gene_sets):
        ax = axes[0, i]  # Get the axis for this subplot
        data = prep_dotplot_data(enr_up.results, gene_set)
        
        if len(data) > 0:
            # Create scatter plot
            # x = gene_ratio (horizontal position)
            # y = range(len(data)) (vertical position - just 0, 1, 2, ...)
            # s = gene_count*30 (dot size proportional to gene count)
            # c = log_pval (color proportional to significance)
            # cmap='YlOrRd' is a color scheme (Yellow-Orange-Red)
            scatter = ax.scatter(data['gene_ratio'], range(len(data)), 
                               s=data['gene_count']*30, c=data['log_pval'], 
                               cmap='YlOrRd', alpha=0.6, edgecolors='black', linewidth=0.5)
            
            # Set y-axis labels to pathway names
            ax.set_yticks(range(len(data)))
            # [:40] truncates long names to 40 characters
            ax.set_yticklabels([t[:40] for t in data['Term']], fontsize=8)
            
            ax.set_xlabel('Gene Ratio', fontsize=10)
            ax.set_title(f'Upregulated - {gene_set.replace("_", " ").replace("2023", "")}', 
                        fontsize=10, fontweight='bold')
            
            # Flip y-axis so top pathway is at top
            ax.invert_yaxis()
            
            # Add a color bar showing what the colors mean
            plt.colorbar(scatter, ax=ax, label='-log10(p)')
        else:
            # If no results, show a message
            ax.text(0.5, 0.5, 'No significant\npathways', 
                   ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f'Upregulated - {gene_set}', fontsize=10)
    
    # DOWNREGULATED GENES (bottom row)
    # Same process, but using different color scheme (Blues for down genes)
    for i, gene_set in enumerate(gene_sets):
        ax = axes[1, i]
        data = prep_dotplot_data(enr_down.results, gene_set)
        
        if len(data) > 0:
            scatter = ax.scatter(data['gene_ratio'], range(len(data)), 
                               s=data['gene_count']*30, c=data['log_pval'], 
                               cmap='Blues', alpha=0.6, edgecolors='black', linewidth=0.5)
            ax.set_yticks(range(len(data)))
            ax.set_yticklabels([t[:40] for t in data['Term']], fontsize=8)
            ax.set_xlabel('Gene Ratio', fontsize=10)
            ax.set_title(f'Downregulated - {gene_set.replace("_", " ").replace("2023", "")}', 
                        fontsize=10, fontweight='bold')
            ax.invert_yaxis()
            plt.colorbar(scatter, ax=ax, label='-log10(p)')
        else:
            ax.text(0.5, 0.5, 'No significant\npathways', 
                   ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f'Downregulated - {gene_set}', fontsize=10)
    
    # Adjust spacing and save
    plt.tight_layout()
    plt.savefig('enrichment_dotplots.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print('✓ Enrichment dot plots saved')
else:
    print('⚠ Enrichment failed, skipping dot plots')

## Summary & Interpretation

**What have we learned?**
- Which genes are consistently changing
- Which biological processes are affected
- How confident we are in these findings

In [ ]:
# Print a comprehensive summary
print('='*80)
print('SUMMARY OF FINDINGS')
print('='*80)

print(f"""
✓ CONSISTENTLY UPREGULATED ({len(up_df)} genes):
  • These genes are progressively induced from d0 → d2 → d5
  • Likely driving the biological transformation (differentiation, activation, etc.)
  • Candidates for functional validation and therapeutic targets
  • Top examples: COL1A1, COL1A2, PAX3, MYLPF
  • Interpretation: These represent the gaining of new function

✓ CONSISTENTLY DOWNREGULATED ({len(down_df)} genes):
  • These genes are progressively suppressed across the time course
  • Likely marking loss of the initial cell state or proliferative capacity
  • Enriched in cell cycle genes (CCNA2, TOP2A, NEK2) — cells exiting proliferation
  • Also enriched in metallothioneins (MT1A, MT1E) — may reflect metabolic shift
  • Interpretation: These represent the loss of old function

⚠ REVERSAL GENES:
  • UP then DOWN ({len(reversal_up_to_down)} genes): Peak expression at d2, then decline
  • DOWN then UP ({len(reversal_down_to_up)} genes): Initially suppressed, then re-induced
  • These suggest temporal regulation — may be transiently needed or transiently repressed
  • Interpretation: Complex regulation with specific timing

📊 PATHWAY INTERPRETATION (from enrichment results above):
  • Upregulated pathways should reflect the target cell state/phenotype
  • Downregulated pathways should reflect the departing cell state (often proliferation/stemness)
  • Use the pathway results to form mechanistic hypotheses

💡 NEXT STEPS:
  1. Validate top genes with qPCR or western blot (pick 3-5 from each category)
  2. Map pathways to known cell types using single-cell RNA-seq markers
  3. Integrate with protein interaction networks (STRING) for mechanistic insights
  4. Consider gene regulation (TF binding sites, miRNA) for the reversal genes
  5. Do functional studies (overexpression/knockdown) on pathway-level genes
""")

print('✓ Analysis complete! All plots saved.')
print('\nGenerated files:')
print('  - volcano_plots.png')
print('  - venn_diagrams.png')
print('  - heatmap_topgenes.png')
print('  - enrichment_dotplots.png')
print('  - consistently_upregulated.csv')
print('  - consistently_downregulated.csv')